<a href="https://colab.research.google.com/github/Alexis0104343/Analisis_de_datos_2026/blob/main/Data_Analyst_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data Analyst Assignment

## Data Challenge

Sales leadership has tasked you with analyzing the data to discover how employee training
impacts sales performance and effectiveness. The goal is to identify patterns and provide
actionable recommendations that drive sales growth and improve team performance by
answering the following questions.
Questions
1. What is the training completion rate for each course by segments (SVP Leader/Region),
factoring in the following caveats? Training is required for all employees except:
* Employees currently on leave are exempt from the training requirement.
* 'Sell More Suite SKU' course is not required for employees within the 'Advocacy'
cost center family.
* “Suite/Automation Technical Lab” and “Advanced Suite Bots Lab Course”
courses are required only for employees in the 'PreSales' and 'Services' cost
center family.

2. How would you analyze the performance of an account executive? Is there a difference
between those who have completed training and those who have not? How would you
segment the data and present your findings to senior stakeholders?
* Hint: Look at this from an overall employee perspective who has completed any
training vs. those who have not completed any training. Any training would count,

rather than distinguishing which specific courses were completed, as they all
contribute to one Suite product.

3. Any other interesting insights that you can see in the data? Any data quality issues with
the data? Any challenges with analyzing the data? What additional data do you think
would be useful for further analyzing the existing datasets?



---

## Data Dictionary

### Table 1 : Employee Information

This table maintains critical employment details for organization members, including
identification, job titles, management hierarchies, tenure, and work region

* **Employee_ID:** Unique identifier for each employee.
* **SVP Leader:** Leader overseeing the employee.
* **Business Title:** Official job title of the employee.
* **Cost Center:** Identifier for the employee's department or unit for cost tracking.
* **Cost Center Family:** Group of related cost centers for financial reporting.
* **Length of Service:** Total time the employee has worked at the company.
* **Leave Status:** Indication of whether the employee is on leave.
* **Is People Manager?:** Indicates if the employee oversees other staff.
* **Region:** Geographic area where the employee works.
* **Manager IC Helper:** Additional data supporting managerial status.
* **IC:** Individual Contributor

### Table 2: Completed Trainings
This table records the professional development activities of employees by linking completed
training programs to their unique identifiers
* **Employee_ID:** Unique identifier linked to an employee who completed the training.
* **Training Name:** Name of the training program or course that the employee completed.

### Table 3: Performance Data
This table includes the sales and revenue generation activities of employees by cataloguing
opportunities, their progression, and financial outcomes. It captures granular data on sales
stage milestones, product-related charges, and revenue figures, all linked by employee and
opportunity identifiers, making it a vital asset for analyzing sales performance and compensation
metrics.

* **Employee_ID:** Unique identifier for the employee associated.
* **Opportunity ID:** Unique identifier for the sales opportunity.
* **Type:** The category or classification of the opportunity.
* Expansion is for existing business

* **Stage 2+ Date:** The date when the opportunity reached or surpassed stage 2 in the
sales process.
* **Stage:** Current stage of the opportunity in the sales pipeline.
 * 02 - Discovery: Initial stage where potential needs and opportunities are
identified with the client.
 * 03 - Solution Review: Potential solutions are presented and reviewed with the
client.
 * 04 - Solution Validation: Client feedback is incorporated, and solutions are
refined and validated.
 * 05 - Contracting / Verba: Terms are negotiated and a verbal agreement may be
reached.
 * 06 - Signed/07 - Closed: Formal agreement is executed with signatures from all
parties. Signed and Closed are counted as finalized
* **Close Date:** The date when the opportunity was closed.
* **Product Rate Plan Charge:** The charge associated with the product's rate plan.
* **Product Name:** The name of the product related to the opportunity.
* **Add-On ARR (converted):** The value of the additional ARR from add-ons,
* **Total Commissionable ARR (converted):** The total annual recurring revenue that is
eligible for commission

#Codigo

##Pregunta 1

In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
df_ass2=pd.read_excel("/content/drive/MyDrive/Data/Assignment_2.xlsx",sheet_name=None)

In [ ]:
df_empl=df_ass2['Employee_Data']
df_train=df_ass2['Completed_Trainings']
df_perf=df_ass2['Performance Data']

In [ ]:
df_empl.head()

,Employee_ID,SVP Leader,Business Title,Cost Center,Cost Center Family,Length of service,Leave Status,is People Manager?,Region,Manager IC Helper
0,1,Leader 1,Senior Commercial Account Executive,532 Commercial AE,Commercial,20,Active,False,EMEA,IC
1,2,Leader 1,Senior Commercial Account Executive,532 Commercial AE,Commercial,13,Active,False,EMEA,IC
2,3,Leader 1,Senior Commercial Account Executive,552 Mid-Market AE,Mid-Market,44,Active,False,EMEA,IC
3,4,Leader 2,Enterprise Corporate Account Executive,508 LATAM Enterprise AE,Enterprise,9,Active,False,LATAM,IC
4,5,Leader 1,Senior Commercial Account Executive,532 Commercial AE,Commercial,15,Active,False,EMEA,IC


In [ ]:
df_empl["Leave Status"].unique()

array(['Active', 'On Leave'], dtype=object)

In [ ]:
df_train.head()

,Employee_ID,Training_Completed
0,2.0,Sell More Suite SKU
1,5.0,Sell More Suite SKU
2,9.0,Sell More Suite SKU
3,10.0,Sell More Suite SKU
4,12.0,Sell More Suite SKU


In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1982 entries, 0 to 1981
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Employee_ID         1782 non-null   float64
 1   Training_Completed  1982 non-null   object 
dtypes: float64(1), object(1)
memory usage: 31.1+ KB


In [ ]:
df_train["Training_Completed"].unique()

array(['Sell More Suite SKU', 'Advanced Suite Bots Lab Course',
       'Suite/Automation Technical Lab'], dtype=object)

In [ ]:
df_tc=df_train.dropna()
df_tc.sort_values(by='Employee_ID')

,Employee_ID,Training_Completed
1981,1.0,Suite/Automation Technical Lab
0,2.0,Sell More Suite SKU
1,5.0,Sell More Suite SKU
2,9.0,Sell More Suite SKU
3,10.0,Sell More Suite SKU
...,...,...
1774,2476.0,Suite/Automation Technical Lab
1775,2478.0,Suite/Automation Technical Lab
1776,2481.0,Sell More Suite SKU
1777,2482.0,Sell More Suite SKU


In [ ]:
df_comb=pd.merge(df_empl,df_train,'inner',"Employee_ID")
df_comb=df_comb.loc[df_comb["Leave Status"]=="Active"]

In [ ]:
conteo_real= pd.crosstab(df_comb['SVP Leader'], df_comb['Training_Completed'])

In [ ]:
import itertools

# 1. Tu conteo real (agregamos .stack() para que sea una serie con índices)
# Luego reseteamos el índice para que 'SVP Leader' y 'Training_Completed' vuelvan a ser columnas
conteo_real = pd.crosstab(df_comb['SVP Leader'], df_comb['Training_Completed']).stack().reset_index(name='Employee_ID')

# 2. Generamos la maestra (esto está perfecto en tu código)
lideres = df_comb['SVP Leader'].unique()
cursos = ['Sell More Suite SKU', 'Suite/Automation Technical Lab', 'Advanced Suite Bots Lab Course']

maestra = pd.DataFrame(list(itertools.product(lideres, cursos)),
                       columns=['SVP Leader', 'Training_Completed'])

# 3. Ahora el merge SÍ encontrará la columna 'Training_Completed'
df_final = pd.merge(maestra, conteo_real, on=['SVP Leader', 'Training_Completed'], how='left')

# 4. Llenamos los ceros
df_final['Employee_ID'] = df_final['Employee_ID'].fillna(0)

,SVP Leader,Training_Completed,Employee_ID
0,Leader 1,Sell More Suite SKU,77
1,Leader 1,Suite/Automation Technical Lab,4
2,Leader 1,Advanced Suite Bots Lab Course,3
3,Leader 4,Sell More Suite SKU,105
4,Leader 4,Suite/Automation Technical Lab,6
5,Leader 4,Advanced Suite Bots Lab Course,2
6,Leader 3,Sell More Suite SKU,114
7,Leader 3,Suite/Automation Technical Lab,2
8,Leader 3,Advanced Suite Bots Lab Course,0
9,Leader 5,Sell More Suite SKU,138


In [ ]:
def es_requerido(row, curso):
    # REGLA GENERAL: Si está "On Leave", NO es requerido (Exento)
    if row['Leave Status'] != 'Active':
        return False

    # REGLA 1: 'Sell More Suite SKU' no es para 'Advocacy'
    if curso == 'Sell More Suite SKU':
        if row['Cost Center Family'] == 'Advocacy':
            return False
        return True

    # REGLA 2: Los "Lab Courses" SOLO son para 'PreSales' y 'Services'
    if curso in ['Suite/Automation Technical Lab', 'Advanced Suite Bots Lab Course']:
        if row['Cost Center Family'] in ['PreSales', 'Services']:
            return True
        return False

    return True # Por defecto, si no cae en excepciones, es requerido

In [ ]:
# Creamos una lista para guardar los conteos de requeridos
registros_requeridos = []

for curso in ['Sell More Suite SKU', 'Suite/Automation Technical Lab', 'Advanced Suite Bots Lab Course']:
    # Filtramos el dataframe original según la función de arriba
    mask = df_comb.apply(lambda x: es_requerido(x, curso), axis=1)
    df_temp = df_comb[mask]

    # Agrupamos por Líder para saber cuántos "obligados" tiene
    conteo = df_temp.groupby('SVP Leader')['Employee_ID'].count().reset_index()
    conteo['Training_Completed'] = curso
    conteo.rename(columns={'Employee_ID': 'Total_Requeridos'}, inplace=True)
    registros_requeridos.append(conteo)

# Unimos todo en una sola tabla de denominadores
df_requeridos = pd.concat(registros_requeridos)

In [ ]:
# Unimos (Merge) las dos tablas
reporte_final = pd.merge(df_final, df_requeridos, on=['SVP Leader', 'Training_Completed'], how='left')

# Si un líder no tiene a nadie requerido para un curso, el merge pondrá NaN. Ponemos 0.
reporte_final['Total_Requeridos'] = reporte_final['Total_Requeridos'].fillna(0)

# CALCULAMOS LA TASA (Evitando división por cero)
reporte_final['Completion_Rate'] = (reporte_final['Employee_ID'] / reporte_final['Total_Requeridos']) * 100

# Limpiamos valores infinitos si Total_Requeridos es 0
reporte_final['Completion_Rate'] = reporte_final['Completion_Rate'].replace([float('inf')], 0).fillna(0)
reporte_final=reporte_final.rename(columns={"Employee_ID":"Completado"})

In [ ]:
import numpy as np

# 1. Primero calculamos la tasa base (manejando el error de división por cero)
# Usamos np.errstate para que no salte un warning molesto en la consola
with np.errstate(divide='ignore', invalid='ignore'):
    reporte_final['Completion_Rate'] = (reporte_final['Completado'] / reporte_final['Total_Requeridos']) * 100

# 2. Reemplazamos los valores donde el denominador era 0 por "N/A"
# .loc[filas, columna] = "N/A"
reporte_final.loc[reporte_final['Total_Requeridos'] == 0, 'Completion_Rate'] = "N/A"

# 3. Para los que SÍ tienen requeridos, redondeamos a 2 decimales y agregamos el %
# Solo aplicamos esto a las filas que NO son "N/A"
mask = reporte_final['Completion_Rate'] != "N/A"
reporte_final.loc[mask, 'Completion_Rate'] = reporte_final.loc[mask, 'Completion_Rate'].astype(float).round(2).astype(str) + "%"

/tmp/ipykernel_312/623404232.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  reporte_final.loc[reporte_final['Total_Requeridos'] == 0, 'Completion_Rate'] = "N/A"


In [ ]:
import pandas as pd
import numpy as np
import itertools

# Definimos los cursos (por si no los tienes en una variable)
cursos = ['Sell More Suite SKU', 'Suite/Automation Technical Lab', 'Advanced Suite Bots Lab Course']

# ==========================================
# 1. CREAR LA TABLA MAESTRA POR REGIÓN
# ==========================================
regiones = df_comb['Region'].unique()
maestra_region = pd.DataFrame(list(itertools.product(regiones, cursos)),
                              columns=['Region', 'Training_Completed'])

# ==========================================
# 2. CONTAR LOS COMPLETADOS POR REGIÓN
# ==========================================
conteo_real_region = pd.crosstab(df_comb['Region'], df_comb['Training_Completed']).stack().reset_index(name='Completado')

# Unimos la maestra con los completados para que aparezcan los ceros
df_final_region = pd.merge(maestra_region, conteo_real_region, on=['Region', 'Training_Completed'], how='left')
df_final_region['Completado'] = df_final_region['Completado'].fillna(0)

# ==========================================
# 3. CALCULAR LOS REQUERIDOS POR REGIÓN
# ==========================================
# (Asegúrate de haber corrido antes la función 'es_requerido' que creamos en los pasos anteriores)
registros_requeridos_region = []

for curso in cursos:
    # Filtramos usando tu función de reglas de negocio
    mask = df_comb.apply(lambda x: es_requerido(x, curso), axis=1)
    df_temp = df_comb[mask]

    # ¡AQUÍ ESTÁ LA MAGIA! Agrupamos por Region en lugar de SVP Leader
    conteo = df_temp.groupby('Region')['Employee_ID'].count().reset_index()
    conteo['Training_Completed'] = curso
    conteo.rename(columns={'Employee_ID': 'Total_Requeridos'}, inplace=True)
    registros_requeridos_region.append(conteo)

df_requeridos_region = pd.concat(registros_requeridos_region)

# ==========================================
# 4. UNIR TODO Y CALCULAR LA TASA CON "N/A"
# ==========================================
reporte_region = pd.merge(df_final_region, df_requeridos_region, on=['Region', 'Training_Completed'], how='left')
reporte_region['Total_Requeridos'] = reporte_region['Total_Requeridos'].fillna(0)

# Calculamos la tasa manejando el error de división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    reporte_region['Completion_Rate'] = (reporte_region['Completado'] / reporte_region['Total_Requeridos']) * 100

# Reemplazamos los infinitos/errores con N/A
reporte_region.loc[reporte_region['Total_Requeridos'] == 0, 'Completion_Rate'] = "N/A"

# Formateamos el resto a porcentaje
mask_region = reporte_region['Completion_Rate'] != "N/A"
reporte_region.loc[mask_region, 'Completion_Rate'] = reporte_region.loc[mask_region, 'Completion_Rate'].astype(float).round(2).astype(str) + "%"

# Mostrar la tabla final
reporte_region

/tmp/ipykernel_312/1549956159.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  reporte_region.loc[reporte_region['Total_Requeridos'] == 0, 'Completion_Rate'] = "N/A"


,Region,Training_Completed,Completado,Total_Requeridos,Completion_Rate
0,EMEA,Sell More Suite SKU,246,485,50.72%
1,EMEA,Suite/Automation Technical Lab,155,233,66.52%
2,EMEA,Advanced Suite Bots Lab Course,117,233,50.21%
3,North America,Sell More Suite SKU,459,832,55.17%
4,North America,Suite/Automation Technical Lab,216,395,54.68%
5,North America,Advanced Suite Bots Lab Course,179,395,45.32%
6,LATAM,Sell More Suite SKU,65,162,40.12%
7,LATAM,Suite/Automation Technical Lab,50,112,44.64%
8,LATAM,Advanced Suite Bots Lab Course,48,112,42.86%
9,APAC,Sell More Suite SKU,24,186,12.9%


#Respuestas

##1

Tabla de tasa de completacion de curso de entrenamiento por lider

In [ ]:
reporte_final

,SVP Leader,Training_Completed,Completado,Total_Requeridos,Completion_Rate
0,Leader 1,Sell More Suite SKU,77,84.0,91.67%
1,Leader 1,Suite/Automation Technical Lab,4,0.0,N/A
2,Leader 1,Advanced Suite Bots Lab Course,3,0.0,N/A
3,Leader 4,Sell More Suite SKU,105,113.0,92.92%
4,Leader 4,Suite/Automation Technical Lab,6,0.0,N/A
5,Leader 4,Advanced Suite Bots Lab Course,2,0.0,N/A
6,Leader 3,Sell More Suite SKU,114,116.0,98.28%
7,Leader 3,Suite/Automation Technical Lab,2,0.0,N/A
8,Leader 3,Advanced Suite Bots Lab Course,0,0.0,N/A
9,Leader 5,Sell More Suite SKU,138,606.0,22.77%


Tabla de tasa de completacion de curso de entrenamiento por region

In [ ]:
reporte_region

,Region,Training_Completed,Completado,Total_Requeridos,Completion_Rate
0,EMEA,Sell More Suite SKU,246,485,50.72%
1,EMEA,Suite/Automation Technical Lab,155,233,66.52%
2,EMEA,Advanced Suite Bots Lab Course,117,233,50.21%
3,North America,Sell More Suite SKU,459,832,55.17%
4,North America,Suite/Automation Technical Lab,216,395,54.68%
5,North America,Advanced Suite Bots Lab Course,179,395,45.32%
6,LATAM,Sell More Suite SKU,65,162,40.12%
7,LATAM,Suite/Automation Technical Lab,50,112,44.64%
8,LATAM,Advanced Suite Bots Lab Course,48,112,42.86%
9,APAC,Sell More Suite SKU,24,186,12.9%


##2

Para evaluar el rendimiento de un ejecutivo me basaria en los siguientes datos
* **Total Commissionable ARR** (quien genera mas ganabcias)
* **Tasa de cierre** (que porcentaje de oportunidades logran cerrar)
* **Total de oportunidades**

##3


* Seria interesante analizar si en verdar hay una correlacion directa entre que un empleado este entrenado y la cantidad de ventas o si es solo casualidad.
* Para poder medir si el curso tuvo un impacto realmente ,una columna que haria falta agregar seria la fecha de realizacion del curso.